In [ ]:
!cat /proc/cpuinfo

In [ ]:
!nvidia-smi

In [ ]:
# Configuração local: nenhum acesso ou montagem de Google Drive.
from pathlib import Path
DATASET_ROOT = Path("./data/faces")
MODELS_ROOT = Path("./models")
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

# Imagens pessoais não são exibidas por padrão. Ative somente em ambiente privado.
SHOW_PRIVATE_IMAGES = False


In [ ]:
pip install tf-keras

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import os.path
import time
import cv2 as cv
import re
import requests
import tempfile
import pylab


from matplotlib.pyplot import imread

from PIL import Image

from sklearn.metrics import confusion_matrix,  multilabel_confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

from keras.models import Sequential
from keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from keras.layers import BatchNormalization, Activation, Dropout
from tf_keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    # Listas para armazenar as imagens das faces, e o nome das pessoas
    X_imagens_teste_predict = []
    Y_categorias_teste_predict = []
    
    c = 0
    
    # Leitura da imagem
    caminho = str(DATASET_ROOT / "treino" / "pessoa_01" / "exemplo.jpg")
    imagem = cv.imread(caminho)
    
    # Conversão de BGR para RGB para visualização correta
    imagem_rgb = cv.cvtColor(imagem, cv.COLOR_BGR2RGB)
    
    
    plt.imshow(imagem_rgb)
    plt.axis("off")  # Opcional: remove os eixos com coordenadas de pixels
    plt.show()


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    im = Image.open(str(DATASET_ROOT / "treino" / "pessoa_01" / "exemplo.jpg"), mode="r")
    
    # This method will show image in any image viewer
    display(im)


In [ ]:
image_sequence = im.getdata()
image_array = np.array(image_sequence)
print(image_array / 255.0)

In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

LARGURA, ALTURA = 250, 250
caminho_base = str(DATASET_ROOT)

def carregar_dados(split_nome):
    X, Y = [], []
    caminho_split = os.path.join(caminho_base, split_nome)

    if not os.path.exists(caminho_split):
        raise FileNotFoundError(f"Diretório não encontrado: {caminho_split}")

    pastas = sorted([p for p in os.listdir(caminho_split) if os.path.isdir(os.path.join(caminho_split, p))])

    for pasta in pastas:
        caminho_pasta = os.path.join(caminho_split, pasta)
        arquivos = [f for f in os.listdir(caminho_pasta) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for nome_arq in arquivos:
            caminho_img = os.path.join(caminho_pasta, nome_arq)
            img = cv.imread(caminho_img)

            if img is None:
                continue

            # Correção de BGR (OpenCV) para RGB
            img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
            img = cv.resize(img, (LARGURA, ALTURA))

            X.append(img)
            Y.append(pasta)

    return np.array(X, dtype=np.float32) / 255.0, Y

# Carregamento limpo dos 3 conjuntos
print("Carregando imagens...")
X_treino, Y_treino_texto = carregar_dados("treino")
X_teste, Y_teste_texto = carregar_dados("teste")
X_validacao, Y_validacao_texto = carregar_dados("validacao")

print(f"Treino: {X_treino.shape} | Teste: {X_teste.shape} | Validação: {X_validacao.shape}")

In [ ]:
# Converte a lista homogênea para array e normaliza entre 0.0 e 1.0
X_imagens_treino = np.array(X_imagens_treino, dtype=np.float32) / 255.0
X_imagens_teste = np.array(X_imagens_teste, dtype=np.float32) / 255.0
X_imagens_validacao = np.array(X_imagens_validacao, dtype=np.float32) / 255.0

print("Formato Treino:", X_imagens_treino.shape)
print("Formato Teste:", X_imagens_teste.shape)
print("Formato Validação:", X_imagens_validacao.shape)

In [ ]:
# Ajusta o codificador APENAS no treino para padronizar os índices de todas as classes
le = LabelEncoder()
le.fit(Y_treino_texto)

classes_nomes = list(le.classes_)
num_classes = len(classes_nomes)
print(f"Pessoas identificadas ({num_classes}): {classes_nomes}")

# Conversão de texto -> número inteiro -> One-Hot Matrix
Y_treino = to_categorical(le.transform(Y_treino_texto), num_classes=num_classes)
Y_teste = to_categorical(le.transform(Y_teste_texto), num_classes=num_classes)
Y_validacao = to_categorical(le.transform(Y_validacao_texto), num_classes=num_classes)

In [ ]:
# Converter Y_categorias_treino, Y_categorias_teste e Y_categorias_validação em um vetor (lista) numpy.
Y_categorias_treino = np.array(Y_categorias_treino)
Y_categorias_teste = np.array(Y_categorias_teste)
Y_categorias_validacao = np.array(Y_categorias_validacao)

In [ ]:
# Binarizar todas as categorias
mlb = MultiLabelBinarizer()
Y_categorias_treino = mlb.fit_transform(Y_categorias_treino)
Y_categorias_teste = mlb.fit_transform(Y_categorias_teste)
Y_categorias_validacao = mlb.fit_transform(Y_categorias_validacao)

In [ ]:
# Verificar o shape da lista X_imagens_treino
X_imagens_treino.shape

In [ ]:
# Verificar o shape da lista X_imagens_teste
X_imagens_teste.shape

In [ ]:
# Verificar o shape da lista X_imagens_validação
X_imagens_validacao.shape


In [ ]:
# Verificar o da lista Y_categorias_treino
Y_categorias_treino.shape

In [ ]:
# Verificar o da lista Y_categorias_teste
Y_categorias_teste.shape

In [ ]:
# Verificar o da lista Y_categorias_validacao
Y_categorias_validacao.shape

In [ ]:
# Conferir se os valores de cada imagem da lista X_imagens_treino foram normalizados
c = 0
for imagem in X_imagens_treino:
  print("O valor da imagem[" + str(c) + "] é:")
  print(imagem)
  c = c + 1

In [ ]:
# Conferir se os valores de cada imagem da lista X_imagens_teste foram normalizados
c = 0
for imagem in X_imagens_teste:
  print("O valor da imagem[" + str(c) + "] é:")
  print(imagem)
  c = c + 1

In [ ]:
# Conferir se os valores de cada imagem da lista X_imagens_validacao foram normalizados
c = 0
for imagem in X_imagens_validacao:
  print("O valor da imagem[" + str(c) + "] é:")
  print(imagem)
  c = c + 1

In [ ]:
# Conferir os valores da categoria Y_categorias_treino de cada imagem
c = 0
for categoria in Y_categorias_treino:
  print("O valor da categoria da imagem[" + str(c) + "] é:")
  print(categoria)
  c = c + 1

In [ ]:
# Conferir os valores da categoria Y_categorias_teste de cada imagem
c = 0
for categoria in Y_categorias_teste:
  print("O valor da categoria da imagem[" + str(c) + "] é:")
  print(categoria)
  c = c + 1

In [ ]:
# Conferir os valores da categoria Y_categorias_validacao de cada imagem
c = 0
for categoria in Y_categorias_validacao:
  print("O valor da categoria da imagem[" + str(c) + "] é:")
  print(categoria)
  c = c + 1

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    #Visualizar a imagem 500 na lista X_imagens_treino e sua respectiva categoria na lista Y_categorias_treino
    plt.imshow(X_imagens_treino[500])
    print(Y_categorias_treino[500])


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    #Visualizar a imagem 500 na lista X_imagens_teste e sua respectiva categoria na lista Y_categorias_teste
    plt.imshow(X_imagens_teste[500])
    print(Y_categorias_teste[500])


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    #Visualizar a imagem 400 na lista X_imagens_validacao e sua respectiva categoria na lista Y_categorias_validacao
    plt.imshow(X_imagens_validacao[350])
    print(Y_categorias_validacao[350])


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    #Visualizar as imagens da lista X_imagens_treino com suas respectivas categorias na lista X_categorias_treino em formato de matriz
    
    #Definir largura e altura da matriz
    Altura_grid = 10
    Largura_grid = 10
    
    
    
    figura, eixos = plt.subplots(Largura_grid, Altura_grid, figsize = (50, 50))
    eixos = eixos.ravel()
    
    tamanhoListaImagens = len(X_imagens_treino)
    
    for i in np.arange(0, Largura_grid * Altura_grid):
      index = np.random.randint(0, tamanhoListaImagens)
      eixos[i].imshow(X_imagens_treino[index])
      eixos[i].set_title(Y_categorias_treino[index])
      eixos[i].axis("off")
    
    plt.subplots_adjust(hspace = 0.4)


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    #Visualizar as imagens da lista X_imagens_teste com suas respectivas categorias na lista X_categorias_teste em formato de matriz
    
    #Definir largura e altura da matriz
    Altura_grid = 10
    Largura_grid = 10
    
    
    
    figura, eixos = plt.subplots(Largura_grid, Altura_grid, figsize = (50, 50))
    eixos = eixos.ravel()
    
    tamanhoListaImagens = len(X_imagens_teste)
    
    for i in np.arange(0, Largura_grid * Altura_grid):
      index = np.random.randint(0, tamanhoListaImagens)
      eixos[i].imshow(X_imagens_teste[index])
      eixos[i].set_title(Y_categorias_teste[index])
      eixos[i].axis("off")
    
    plt.subplots_adjust(hspace = 0.4)


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    #Visualizar as imagens da lista X_imagens_validacao com suas respectivas categorias na lista X_categorias_validacao em formato de matriz
    
    #Definir largura e altura da matriz
    Altura_grid = 10
    Largura_grid = 10
    
    
    
    figura, eixos = plt.subplots(Largura_grid, Altura_grid, figsize = (50, 50))
    eixos = eixos.ravel()
    
    tamanhoListaImagens = len(X_imagens_validacao)
    
    for i in np.arange(0, Largura_grid * Altura_grid):
      index = np.random.randint(0, tamanhoListaImagens)
      eixos[i].imshow(X_imagens_validacao[index])
      eixos[i].set_title(Y_categorias_validacao[index])
      eixos[i].axis("off")
    
    plt.subplots_adjust(hspace = 0.4)


Treinar Modelo

O modelo consiste nas seguintes camadas:

Primeira camada: A PRIMEIRA CAMADA CONVOLUCIONAL
Entrada: 500x500x3
Saída: 496x496x6
Cálculo: (Entrada - Filtro + 1) / Stride => (500 - 5 + 1) / 1 = 496
Utilizando 6 filtros de dimensão 5x5 aplicados sobre os 3 canais de cor da entrada.
Aplicação da função de ativação ReLU na saída.
Subamostragem (Pooling): Entrada 496x496x6 e saída 248x248x6 (redução espacial com janela 2x2 e stride=2).

Segunda camada: A SEGUNDA CAMADA CONVOLUCIONAL
Entrada: 248x248x6
Saída: 244x244x16
Cálculo: (Entrada - Filtro + 1) / Stride => (248 - 5 + 1) / 1 = 244
Utilizando 16 filtros de dimensão 5x5 sobre os 6 canais anteriores.
Aplicação da função de ativação ReLU na saída.
Subamostragem (Pooling): Entrada 244x244x16 e saída 122x122x16 (redução espacial com janela 2x2 e stride=2).

PASSO 3: ACHATAMENTO DA REDE (FLATTEN)
Entrada: 122x122x16
Saída: 238.144 neurônios (122 * 122 * 16 = 238.144).

PASSO 4: PRIMEIRA CAMADA TOTALMENTE CONECTADA (DENSE)
Entrada: 238.144 neurônios
Saída: 120 neurônios
Aplicação da função de ativação ReLU na saída e camada de Dropout (0.3) para controle de sobreajuste.

PASSO 5: SEGUNDA CAMADA TOTALMENTE CONECTADA (DENSE)
Entrada: 120 neurônios
Saída: 84 neurônios
Aplicação da função de ativação ReLU na saída.

PASSO 6: CAMADA DE SAÍDA (DENSE)
Entrada: 84 neurônios
Saída: 6 neurônios (correspondentes às classes: pessoa_01, pessoa_02, pessoa_03, pessoa_04, pessoa_05, pessoa_06)
Aplicação da função de ativação Softmax para distribuição de probabilidades das 6 classes mutuamente exclusivas.

In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

# 1. Configurações de dimensão e caminho base
LARGURA, ALTURA = 500, 500
caminho_base = str(DATASET_ROOT)

def carregar_dados(split_nome):
    X, Y = [], []
    caminho_split = os.path.join(caminho_base, split_nome)

    if not os.path.exists(caminho_split):
        raise FileNotFoundError(f"Diretório não encontrado: {caminho_split}")

    pastas = sorted([p for p in os.listdir(caminho_split) if os.path.isdir(os.path.join(caminho_split, p))])

    for pasta in pastas:
        caminho_pasta = os.path.join(caminho_split, pasta)
        arquivos = [f for f in os.listdir(caminho_pasta) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for nome_arq in arquivos:
            caminho_img = os.path.join(caminho_pasta, nome_arq)
            img = cv.imread(caminho_img)

            if img is None:
                continue

            # Correção de BGR (padrão OpenCV) para RGB
            img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
            img = cv.resize(img, (LARGURA, ALTURA))

            X.append(img)
            Y.append(pasta)

    return np.array(X, dtype=np.float32) / 255.0, Y

# 2. Carregamento das imagens
print("Carregando datasets com resolução 500x500...")
X_treino, Y_treino_texto = carregar_dados("treino")
X_teste, Y_teste_texto = carregar_dados("teste")
X_validacao, Y_validacao_texto = carregar_dados("validacao")

print(f"Treino: {X_treino.shape} | Teste: {X_teste.shape} | Validação: {X_validacao.shape}")

# 3. Codificação para as 6 pessoas identificadas
le = LabelEncoder()
le.fit(Y_treino_texto)

classes_nomes = [str(cls) for cls in le.classes_]
num_classes = len(classes_nomes)
print(f"Pessoas identificadas ({num_classes}): {classes_nomes}")

# Conversão em matrizes One-Hot de 6 colunas
Y_treino = to_categorical(le.transform(Y_treino_texto), num_classes=num_classes)
Y_teste = to_categorical(le.transform(Y_teste_texto), num_classes=num_classes)
Y_validacao = to_categorical(le.transform(Y_validacao_texto), num_classes=num_classes)

# 4. Arquitetura LeNet ajustada para 500x500 e 6 classes
LeNet = models.Sequential([
    layers.Input(shape=(LARGURA, ALTURA, 3)),

    # Bloco Convolucional 1: Saída (496, 496, 6) -> Pooling: (248, 248, 6)
    layers.Conv2D(6, (5, 5), activation='relu'),
    layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),

    # Bloco Convolucional 2: Saída (244, 244, 16) -> Pooling: (122, 122, 16)
    layers.Conv2D(16, (5, 5), activation='relu'),
    layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),

    # Achatamento (122 * 122 * 16 = 238.144 neurônios)
    layers.Flatten(),

    # Camadas Densas
    layers.Dense(120, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(84, activation='relu'),

    # Saída Multiclasse (6 classes)
    layers.Dense(num_classes, activation='softmax')
])

LeNet.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

LeNet.summary()

# 5. Checkpoint e Treinamento
checkpoint = ModelCheckpoint(
    str(MODELS_ROOT / "melhor_modelo_lenet_6classes.keras"),
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)

history = LeNet.fit(
    X_treino, Y_treino,
    batch_size=8,
    epochs=15,
    validation_data=(X_validacao, Y_validacao),
    callbacks=[checkpoint],
    verbose=1
)

# 6. Avaliação e Matriz de Confusão no Teste
y_pred_probs = LeNet.predict(X_teste)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(Y_teste, axis=1)

print("\n=== Relatório de Classificação ===")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(num_classes)),
    target_names=classes_nomes,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes_nomes,
    yticklabels=classes_nomes
)
plt.title('Matriz de Confusão - Reconhecimento Facial (6 Pessoas)')
plt.xlabel('Predição')
plt.ylabel('Classe Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
LeNet.compile(optimizer = tf.optimizers.Adam(), loss = tf.keras.losses.binary_crossentropy , metrics = ['accuracy'])

In [ ]:
history = LeNet.fit(X_imagens_treino,
                 Y_categorias_treino,
                 batch_size = 1,
                 epochs = 4,
                 verbose = 1,
                 validation_data = (X_imagens_teste, Y_categorias_teste))

In [ ]:
test_loss, test_acc = LeNet.evaluate(X_imagens_teste, Y_categorias_teste)
print('\nTest accuracy: {}'.format(test_acc))

In [ ]:
score = LeNet.evaluate(X_imagens_treino, Y_categorias_treino)
print('Test Accuracy: {}'.format(score[1]))

In [ ]:
history.history.keys()

In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
epochs = range(len(accuracy))

plt.plot(epochs, accuracy, 'bo', label='Training Accuracy')
plt.plot(epochs, val_accuracy, 'b', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

In [ ]:
plt.plot(epochs, loss, 'ro', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Extrai a classe real a partir da matriz One-Hot
y_true = np.argmax(Y_categorias_teste, axis=1)

# 2. Predição do modelo
y_pred = LeNet.predict(X_imagens_teste)
y_pred_classes = np.argmax(y_pred, axis=1)

# 3. Nomes das classes para os gráficos e relatórios
# Substitua pela sua lista de pastas se tiver, ou mantenha numérico de 0 a 15:
total_classes = Y_categorias_teste.shape[1]
classes_nomes = [f"Pessoa_{i}" for i in range(total_classes)]

# 4. Imprime o relatório detalhado de métricas por classe
print("=== Relatório de Classificação ===")
print(classification_report(y_true, y_pred_classes, target_names=classes_nomes))

# 5. Plota a Matriz de Confusão em formato de mapa de calor
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes_nomes,
    yticklabels=classes_nomes
)
plt.xlabel('Predição do Modelo')
plt.ylabel('Classe Real')
plt.title('Matriz de Confusão no Conjunto de Teste')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
predicted_classes = LeNet.predict_classes(X_imagens_treino)
y_true = Y_categorias_treino

In [ ]:
classes_previstas_treino = (LeNet.predict(X_imagens_treino) > 0.5).astype("int32")

In [ ]:
classes_previstas_validacao = (LeNet.predict(X_imagens_validacao) > 0.5).astype("int32")

In [ ]:
classes_previstas_validacao

In [ ]:
classes_previstas_validacao.shape

In [ ]:
classes_previstas_validacao[420]

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    plt.imshow(X_imagens_validacao[420])


In [ ]:
classes_previstas

In [ ]:
classes_previstas.shape

In [ ]:
classes_previstas[10]

In [ ]:
Y_categorias_treino[10]

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    L = 7
    W = 7
    
    fig, axes = plt.subplots(L, W, figsize = (34, 34))
    axes = axes.ravel()
    
    for i in np.arange(0, L*W):
        axes[i].imshow(X_imagens_treino[i])
        axes[i].set_title('Prediction = {}\n True = {}'.format(classes_previstas[i], Y_categorias_treino[i]))
        axes[i].axis('off')
    
    plt.subplots_adjust(wspace = 1)


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    L = 7
    W = 7
    
    fig, axes = plt.subplots(L, W, figsize = (34, 34))
    axes = axes.ravel()
    
    for i in np.arange(0, L*W):
        axes[i].imshow(X_imagens_treino[ 3000 + i])
        axes[i].set_title('Prediction = {}\n True = {}'.format(classes_previstas[3000 + i], Y_categorias_treino[3000 + i]))
        axes[i].axis('off')
    
    plt.subplots_adjust(wspace = 1)


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    L = 7
    W = 7
    
    fig, axes = plt.subplots(L, W, figsize = (34, 34))
    axes = axes.ravel()
    
    for i in np.arange(0, L*W):
        axes[i].imshow(X_imagens_treino[ 700 + i])
        axes[i].set_title('Prediction = {}\n True = {}'.format(classes_previstas[700 + i], Y_categorias_treino[700 + i]))
        axes[i].axis('off')
    
    plt.subplots_adjust(wspace = 1)


In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    L = 7
    W = 7
    
    fig, axes = plt.subplots(L, W, figsize = (34, 34))
    axes = axes.ravel()
    
    for i in np.arange(0, L*W):
        axes[i].imshow(X_imagens_treino[ 1500 + i])
        axes[i].set_title('Prediction = {}\n True = {}'.format(classes_previstas[1500 + i], Y_categorias_treino[1500 + i]))
        axes[i].axis('off')
    
    plt.subplots_adjust(wspace = 1)


In [ ]:
classes_previstas[3000]

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    plt.imshow(X_imagens_treino[4000])
    print("Categoria verdadeira da imagem é: {}" .format(Y_categorias_treino[4000]))
    print("Categoria prevista da imagem é:   {}" .format(classes_previstas[4000]))


Predict on new data
Finally, let's use our model to classify an image that wasn't included in the training or validation sets.

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    sunflower_path = str(DATASET_ROOT / "inferencia" / "exemplo.jpg")
    
    Lista_imagem = []
    
    print(sunflower_path)
    
    # Ler imagem usando cv2
    imagem = cv.imread(sunflower_path)
    
    # Redimensionar a imagem usando cv2
    imagem = cv.resize(imagem, (250, 250))
    
    print(imagem.shape)
    
    img = tf.keras.preprocessing.image.load_img(
        sunflower_path, target_size=(250, 250)
    )
    
    
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) # Create a batch
    
    imagem_array = tf.keras.preprocessing.image.img_to_array(imagem)
    imagem_array = tf.expand_dims(imagem_array, 0) # Create a batch
    
    
    
    
    
    
    print(img_array.shape)
    
    
    # Como a imagem que está sendo exibida é super estranha, o fato da rede estar dando um resultado baixo na predição deve ser por conta desse problema.
    # Portanto, para resolver esse problema, preciso garantir que o método predict irá receber a imagem tal como deve.
    img = np.array(img)
    Lista_imagem.append(img)
    
    
    Lista_imagem = np.array(Lista_imagem)
    
    print("EOQ")
    
    plt.imshow(Lista_imagem[0])
    
    
    print(imagem_array.shape)
    
    predictions = LeNet.predict(X_imagens_validacao)
    score = tf.nn.softmax(predictions[0])
    
    print(np.argmax(score))
    
    print(
        "This image most likely belongs to {} with a {:.2f} percent confidence."
        .format(Y_categorias_treino[np.argmax(score)], 100 * np.max(score))
    )


In [ ]:
X_imagens_teste_predict.clear

In [ ]:
X_imagens_teste_predict = np.array(X_imagens_teste_predict)

In [ ]:
X_imagens_teste_predict.shape

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    plt.imshow(X_imagens_teste_predict[5])


In [ ]:
# Let's obtain a temporary storage directory
MODEL_DIR = tempfile.gettempdir()

In [ ]:
MODEL_DIR

In [ ]:
# Let's specify the model version, choose #1 for now
version = 1

In [ ]:
# Let's join the temp model directory with our chosen version number
# The expected result will be = '\tmp\version number'
export_path = os.path.join(MODEL_DIR, str(version))
print('export_path = {}\n'.format(export_path))

In [ ]:
# Let's save the model using simple_save
# If the directory already exists, we will remove it using '!rm'
# rm removes each file specified on the command line.

if os.path.isdir(export_path):
  print('\nAlready saved a model, cleaning up\n')
  !rm -r {export_path}

tf.saved_model.save(
    tf.compat.v1.Session()
    export_path,
    inputs={'input_image': cnn.input},
    outputs={t.name:t for t in cnn.outputs})


In [ ]:
model = Sequential()
# O método Sequential() significa que o modelo, a rede, será montada camada por camada. O que na prática significa montar um arrayList contendo

model.add(Conv2D(32, 3, padding = 'same', input_shape = input_shape, kernel_initializer = 'he_normal', activation = 'relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(3))
model.add(Dropout(0.25))

model.add(Conv2D(64, 3, padding = 'same', kernel_initializer = 'he_normal', activation = 'relu'))
model.add(BatchNormalization())
model.add(Conv2D(64, 3, padding = 'same', kernel_initializer = 'he_normal', activation = 'relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())
model.add(Dropout(0.25))

model.add(Conv2D(128, 3, padding = 'same', kernel_initializer = 'he_normal', activation = 'relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, 2, padding = 'same', kernel_initializer = 'he_normal', activation = 'relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(1024, activation = 'relu', kernel_initializer = 'he_normal'))
model.add(BatchNormalization())
model.add(Dropout(0.5))


model.add(Dense(len(mlb.classes_), activation = 'sigmoid'))


checkpoint = ModelCheckpoint(str(MODELS_ROOT / "best_model.h5"), save_best_only = True, monitor = 'val_loss', verbose = 1)

model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

history = model.fit_generator(datagen.flow(train_x, train_y, batch_size = 64),
                             validation_data = (val_x, val_y), epochs = 5, verbose = 1, callbacks = [checkpoint])

In [ ]:
model = Sequential([
  data_augmentation,
  layers.experimental.preprocessing.Rescaling(1./255),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes)
])

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
epochs = 15
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
sunflower_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"
sunflower_path = tf.keras.utils.get_file('Red_sunflower', origin=sunflower_url)

img = keras.preprocessing.image.load_img(
    sunflower_path, target_size=(img_height, img_width)
)
img_array = keras.preprocessing.image.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = tf.nn.softmax(predictions[0])

print(
    "This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

In [ ]:
LeNet.predict(X_imagens_validacao)

In [ ]:
score = model.evaluate(test_x, test_y)
print('Test Accuracy: {}'.format(score[1]))

In [ ]:
history.history.keys()

In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
epochs = range(len(accuracy))

plt.plot(epochs, accuracy, 'bo', label='Training Accuracy')
plt.plot(epochs, val_accuracy, 'b', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

In [ ]:
plt.plot(epochs, loss, 'ro', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()

In [ ]:
predicted_classes = model.predict_classes(test_x)
y_true = test_y

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, predicted_classes)
plt.figure(figsize = (25, 25))
sns.heatmap(cm, annot = True)

In [ ]:
# Visualização de dados pessoais desativada por padrão.
if SHOW_PRIVATE_IMAGES:
    L = 7
    W = 7
    
    fig, axes = plt.subplots(L, W, figsize = (24, 24))
    axes = axes.ravel()
    
    for i in np.arange(0, L*W):
        axes[i].imshow(test_x[i])
        axes[i].set_title("Prediction = {}\n True = {}".format(predicted_classes[i], y_true[i]))
        axes[i].axis("off")
    
    plt.subplots_adjust(wspace = 1)


In [ ]:
pip install tensorflowjs

In [ ]:
import tensorflowjs

In [ ]:
tensorflowjs.converters.save_keras_model(model, "models")